# 01 - People, Names, and Roles

                This notebook creates the article-person table, name-disambiguation diagnostics, and editor-role templates. The outputs are designed to make uncertainty visible before any actor-level interpretation is attempted.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from hsr_analysis.common import *

import numpy as np
import pandas as pd

ensure_project_structure(ROOT)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {ROOT}")

In [ ]:
articles_path = ROOT / "outputs/tables/articles_clean.csv"
if not articles_path.exists():
    raw, _ = load_article_source(ROOT)
    articles = build_articles_clean(raw)
    write_csv(articles, articles_path)
else:
    articles = pd.read_csv(articles_path)

corrections = load_name_corrections(ROOT)
article_persons = build_article_persons(articles, corrections)
write_csv(article_persons, ROOT / "outputs/tables/article_persons.csv")

variants, possible_duplicates, top_manual = name_diagnostics(article_persons)
write_csv(variants, ROOT / "outputs/tables/name_variants_detected.csv")
write_csv(possible_duplicates, ROOT / "outputs/tables/possible_duplicate_persons.csv")
write_csv(top_manual, ROOT / "outputs/tables/top_persons_manual_check.csv")

persons = (
    article_persons.groupby(["person_id", "person_name"])
    .agg(
        n_articles=("article_id", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        n_raw_name_variants=("person_raw", "nunique"),
        raw_name_variants=("person_raw", lambda x: " | ".join(sorted(set(map(compact_ws, x))))),
    )
    .reset_index()
)
persons["active_span"] = persons["last_year"] - persons["first_year"] + 1

roles_path = ROOT / "data/editors_roles.csv"
roles = pd.read_csv(roles_path)
if roles.empty:
    for col in ["is_editor_any", "is_managing_editor", "is_special_issue_editor", "is_board_member"]:
        persons[col] = False
    persons["editor_role_count"] = 0
    persons["editor_active_start"] = pd.NA
    persons["editor_active_end"] = pd.NA
    persons["editor_role_certainty"] = "unknown"
else:
    roles["person_name"] = roles["person_name"].map(compact_ws)
    role_summary = (
        roles.groupby("person_name")
        .agg(
            editor_role_count=("role_type", "count"),
            editor_active_start=("start_year", "min"),
            editor_active_end=("end_year", "max"),
            editor_role_certainty=("certainty", lambda x: "; ".join(sorted(set(map(compact_ws, x))) or ["unknown"])),
            role_types=("role_type", lambda x: " | ".join(sorted(set(map(compact_ws, x))))),
        )
        .reset_index()
    )
    persons = persons.merge(role_summary, on="person_name", how="left")
    persons["editor_role_count"] = persons["editor_role_count"].fillna(0).astype(int)
    persons["is_editor_any"] = persons["editor_role_count"] > 0
    persons["is_managing_editor"] = persons.get("role_types", "").fillna("").str.contains("managing_editor", regex=False)
    persons["is_special_issue_editor"] = persons.get("role_types", "").fillna("").str.contains("special_issue_editor|issue_editor|guest_editor", regex=True)
    persons["is_board_member"] = persons.get("role_types", "").fillna("").str.contains("editorial_board|consulting_editor|cooperating_editor", regex=True)
    persons["editor_role_certainty"] = persons["editor_role_certainty"].fillna("unknown")

write_csv(persons, ROOT / "outputs/tables/persons_clean.csv")

diagnostic_md = [
    "# Name and Role Diagnostics",
    "",
    f"- Article-person rows: {len(article_persons):,}",
    f"- Distinct normalized persons: {persons['person_id'].nunique():,}",
    f"- Persons with multiple raw variants: {int((variants.get('n_name_variants', pd.Series(dtype=int)) > 1).sum()) if not variants.empty else 0:,}",
    f"- Possible duplicate name groups: {len(possible_duplicates):,}",
    f"- Editor-role rows supplied manually: {len(roles):,}",
    "",
    "The top-50 recurring persons are written to `outputs/tables/top_persons_manual_check.csv` for manual review.",
]
write_markdown("\n".join(diagnostic_md), ROOT / "outputs/diagnostics/name_and_role_diagnostics.md")

display(persons.sort_values("n_articles", ascending=False).head(15))
display(possible_duplicates.head(15))